# DEM solver 선택 실행 PoC — v1: 사각형 ROI (현재 버전)

`0_Test_set`의 `.npz` 또는 IDL `.sav` 프레임을 확장자에 맞게 읽고, `1_Resp`의 AIA Temperature Response Function을 적용해 `2_Solver`의 여러 DEM 구현 중 선택한 방법을 실행한다. ROI는 설정 셀의 `ROI_CENTER_XY`/`ROI_SIZE` 사각형으로 고정한다. 출력 그림은 **log-scale DEM profile 한 장**뿐이다.

> **응답 출처와 제한:** `1_Resp/aia_temperature_response.npz`는 SunPy/aiapy 기반 `synthesizAR`가 배포하는 SolarSoft `aia_get_response(/temp,/dn)` 결과를 NumPy 형식으로 변환한 것이다. 실제 AIA 온도응답이지만 `/timedepend_date`, `/evenorm`, `/chiantifix`는 적용되지 않았다. 따라서 현재 그래프는 solver 비교용이며 관측시각 보정까지 포함한 정량 결과는 아니다.

In [ ]:
from pathlib import Path
import contextlib
import io
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import readsav


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '0_Test_set').is_dir() and (candidate / '2_Solver').is_dir():
            return candidate
    raise FileNotFoundError('0_Test_set과 2_Solver를 포함한 저장소 루트를 찾지 못했습니다.')


REPO_ROOT = find_repo_root()
SOLVER_DIRS = [
    REPO_ROOT / '2_Solver/simple_reg_dem/python',
    REPO_ROOT / '2_Solver/demreg/python_ref',
    REPO_ROOT / '2_Solver/dem_sites/python',
    REPO_ROOT / '2_Solver/cheung_sparse_em/python',
    REPO_ROOT / '2_Solver/aschwanden_aia_teem/python',
]
for solver_dir in reversed(SOLVER_DIRS):
    solver_path = str(solver_dir)
    if solver_path not in sys.path:
        sys.path.insert(0, solver_path)

from simple_reg_dem import simple_reg_dem
from dn2dem_pos import dn2dem_pos
from dem_sites import dem_sites, gaussian_function
from aia_sparse_em import aia_sparse_em_init, aia_sparse_em_solve
from aia_teem import build_flux_table, fit as aia_teem_fit

print(f'Repository: {REPO_ROOT}')

## 설정

- `DATA_FILE`을 `.npz` 또는 `.sav`로 바꾸면 같은 로더 인터페이스를 사용한다. 두 파일은 현재 동일 프레임의 포맷 복제본이다.
- `RESPONSE_FILE`은 `1_Resp`에 저장한 실제 AIA 온도응답 NPZ다. 관측 채널 순서를 검사한 뒤 solver 온도격자로 보간한다.
- 파일에는 intensity 단위 메타데이터가 없다. 기본값은 `rate`(DN/s로 가정)이며 원자료가 노출시간 동안의 DN count라면 `INTENSITY_KIND = 'counts'`로 바꾼다. 자동 판정하지 않는다.
- ROI는 `ROI_CENTER_XY` 중심, 한 변 `ROI_SIZE` 픽셀의 사각형이다. contour로 직접 그리고 싶으면 v2 노트북(`DEM_solver_PoC_v2_contour.ipynb`)을 사용한다.
- AIA 6채널로 완전한 역산까지 가능한 Python solver 5개를 기본 선택한다. `SELECTED_SOLVERS`에서 원하는 방법만 남기면 된다.
- `CHEUNG_ENGINE='idl'`은 IDL SIMPLEX의 성공·비수렴 결과까지 재현하므로 느리고 `status=3`이 많이 나올 수 있다. 빠르고 강건한 SciPy 해가 필요하면 `'highs'`로 바꿀 수 있지만 IDL 수치 동등 결과는 아니다.

In [ ]:
DATA_FILE = REPO_ROOT / '0_Test_set/frame_2011-02-15T014950000.npz'
RESPONSE_FILE = REPO_ROOT / '1_Resp/aia_temperature_response.npz'

INTENSITY_KIND = 'rate'  # 'rate' 또는 'counts'
ROI_CENTER_XY = (512, 512)
ROI_SIZE = 32

SELECTED_SOLVERS = (
    'simple_reg_dem',
    'demreg',
    'dem_sites',
    'cheung_sparse_em',
    'aschwanden_aia_teem',
)

SYSTEMATIC_FRACTION = 0.10
SITES_RESPONSE_ERROR = 0.20
CHEUNG_ENGINE = 'idl'  # 'idl': IDL parity, 'highs': 더 강건하지만 parity 아님
CHEUNG_ITMAX = 5000  # IDL parity 값; 낮추면 status와 결과가 달라질 수 있음
DEMREG_BATCH_PIXELS = 128  # Jupyter/Windows에서 내부 multiprocessing 분기를 피하는 안전한 batch

In [ ]:
REQUIRED_KEYS = {'intensity', 'wavelengths', 't_rec', 'exptime'}


def _decode_scalar_text(value):
    item = np.asarray(value).item()
    if isinstance(item, (bytes, np.bytes_)):
        return item.decode('utf-8', errors='replace')
    return str(item)


def _read_container(path):
    suffix = path.suffix.lower()
    if suffix == '.npz':
        with np.load(path, allow_pickle=False) as archive:
            return {key.lower(): np.array(archive[key], copy=True) for key in archive.files}
    if suffix == '.sav':
        loaded = readsav(path, python_dict=True, verbose=False)
        return {str(key).lower(): np.array(value, copy=True) for key, value in loaded.items()}
    raise ValueError(f'지원하지 않는 형식입니다: {suffix} (지원: .npz, .sav)')


def load_frame(path):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)

    raw = _read_container(path)
    missing = REQUIRED_KEYS - set(raw)
    if missing:
        raise KeyError(f'{path.name}에 필수 key가 없습니다: {sorted(missing)}')

    wavelengths = np.ascontiguousarray(raw['wavelengths'], dtype=np.int64).reshape(-1)
    exptime = np.ascontiguousarray(raw['exptime'], dtype=np.float64).reshape(-1)
    intensity = np.ascontiguousarray(raw['intensity'], dtype=np.float64)
    n_channels = wavelengths.size

    candidate_axes = [axis for axis, size in enumerate(intensity.shape) if size == n_channels]
    if len(candidate_axes) != 1:
        raise ValueError(
            f'채널 축을 하나로 결정할 수 없습니다: shape={intensity.shape}, n_channels={n_channels}'
        )
    cube = np.ascontiguousarray(np.moveaxis(intensity, candidate_axes[0], -1))

    if cube.ndim != 3 or cube.shape[-1] != n_channels:
        raise ValueError(f'예상한 (y, x, channel) 3차원 배열이 아닙니다: {cube.shape}')
    if exptime.size != n_channels or np.any(exptime <= 0):
        raise ValueError('exptime은 채널 수와 같고 모두 양수여야 합니다.')
    if not np.all(np.isfinite(cube)) or not np.all(np.isfinite(exptime)):
        raise ValueError('intensity/exptime에 유한하지 않은 값이 있습니다.')

    return {
        'path': path,
        'intensity': cube,
        'wavelengths': wavelengths,
        'exptime': exptime,
        't_rec': _decode_scalar_text(raw['t_rec']),
    }

In [ ]:
frame = load_frame(DATA_FILE)

print(f"Loaded: {frame['path'].name}")
print(f"t_rec: {frame['t_rec']}")
print(f"canonical shape: {frame['intensity'].shape} = (y, x, channel)")
print(f"wavelengths [Å]: {frame['wavelengths'].tolist()}")

## 픽셀별 관측값과 AIA temperature response

ROI 안 모든 픽셀을 각각 관측 벡터로 사용한다 (mean(DEM(픽셀들)) 방식 — DEM 역산이 비선형이라
ROI를 먼저 평균하는 것과 결과가 다르다). 음수 intensity는 원본 로더에서 보존하고, solver에
전달할 값에서만 작은 양수로 제한한다. 오차는 픽셀별 Poisson 항과 10% systematic floor를
제곱합으로 둔 보수적 PoC 모델이다.

저장된 AIA 온도응답의 원본 격자를 읽고 관측 채널 순서에 맞춘 뒤, solver 공통 격자에
log-temperature 기준으로 보간한다. 응답함수 자체는 그리지 않는다.

In [ ]:
cube = frame['intensity']
exptime = frame['exptime']
wavelengths = frame['wavelengths']

if INTENSITY_KIND == 'rate':
    rate_cube = cube.copy()
elif INTENSITY_KIND == 'counts':
    rate_cube = cube / exptime[None, None, :]
else:
    raise ValueError("INTENSITY_KIND는 'rate' 또는 'counts'여야 합니다.")

# ROI 마스크 — 설정 셀의 사각형.
center_x, center_y = map(int, ROI_CENTER_XY)
if ROI_SIZE <= 0:
    raise ValueError('ROI_SIZE는 양수여야 합니다.')
x0 = center_x - ROI_SIZE // 2
x1 = x0 + ROI_SIZE
y0 = center_y - ROI_SIZE // 2
y1 = y0 + ROI_SIZE
if not (0 <= x0 < x1 <= rate_cube.shape[1] and 0 <= y0 < y1 <= rate_cube.shape[0]):
    raise ValueError(f'ROI가 영상 범위를 벗어납니다: x={x0}:{x1}, y={y0}:{y1}')
roi_mask = np.zeros(rate_cube.shape[:2], dtype=bool)
roi_mask[y0:y1, x0:x1] = True
roi_label = f'box x={x0}:{x1}, y={y0}:{y1}'

if not np.any(roi_mask):
    raise ValueError('ROI 안에 픽셀이 없습니다.')

roi_raw = rate_cube[roi_mask][:, None, :]           # (npix, 1, channel) 픽셀별 DN/s
positive_floor = max(float(np.max(np.abs(roi_raw))) * 1e-8, 1e-6)
roi_rate = np.maximum(roi_raw, positive_floor)
roi_counts = roi_rate * exptime
roi_sigma_counts = np.sqrt(
    np.maximum(roi_counts, 1.0) + (SYSTEMATIC_FRACTION * roi_counts) ** 2
)
roi_sigma_rate = roi_sigma_counts / exptime

def load_temperature_response(path, target_logt, observation_wavelengths):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as archive:
        required = {'logt', 'wavelengths', 'response', 'response_unit'}
        missing = required - set(archive.files)
        if missing:
            raise KeyError(f'{path.name}에 필수 key가 없습니다: {sorted(missing)}')
        source_logt = np.asarray(archive['logt'], dtype=np.float64)
        source_wavelengths = np.asarray(archive['wavelengths'], dtype=np.int64)
        source_response = np.asarray(archive['response'], dtype=np.float64)
        metadata = {
            key: archive[key].item()
            for key in ('response_unit', 'source_url', 'source_sha256',
                        'upstream_generation', 'time_dependent_correction',
                        'eve_normalization', 'sunpy_version', 'aiapy_version')
            if key in archive.files
        }

    if source_response.shape != (source_logt.size, source_wavelengths.size):
        raise ValueError(f'응답 shape 불일치: {source_response.shape}')
    if not np.all(np.diff(source_logt) > 0):
        raise ValueError('응답 logT 격자는 엄격히 증가해야 합니다.')
    if target_logt[0] < source_logt[0] or target_logt[-1] > source_logt[-1]:
        raise ValueError('solver logT 격자가 응답함수 범위를 벗어납니다.')
    if not np.all(np.isfinite(source_response)) or np.any(source_response < 0):
        raise ValueError('응답함수는 유한한 비음수 값이어야 합니다.')

    channel_indices = []
    for wavelength in observation_wavelengths:
        matches = np.flatnonzero(source_wavelengths == wavelength)
        if matches.size != 1:
            raise KeyError(f'응답함수에 {wavelength} Å 채널이 정확히 하나 있어야 합니다.')
        channel_indices.append(int(matches[0]))
    response = np.column_stack([
        np.interp(target_logt, source_logt, source_response[:, index])
        for index in channel_indices
    ])
    return response, metadata


logt = np.linspace(5.5, 7.5, 41, dtype=np.float64)
temperature_response, response_metadata = load_temperature_response(
    RESPONSE_FILE, logt, wavelengths
)

print(f'ROI: {roi_label}, pixels={np.count_nonzero(roi_mask)}')
print(f'Input interpretation: {INTENSITY_KIND}')
print(' channel   mean rate   mean sigma_rate    exptime')
for k, wavelength in enumerate(wavelengths):
    print(f'{wavelength:>4} Å  {roi_rate[..., k].mean():10.4g}  '
          f'{roi_sigma_rate[..., k].mean():14.4g}  {exptime[k]:9.4f} s')
print(f'logT grid: {logt[0]:.2f} .. {logt[-1]:.2f} ({logt.size} points)')
print(f'Response file: {Path(RESPONSE_FILE).name}')
print(f"Response unit: {response_metadata.get('response_unit', 'unknown')}")
print(f"Response source: {response_metadata.get('upstream_generation', 'unknown')}")
print(f"SunPy/aiapy: {response_metadata.get('sunpy_version', '?')} / {response_metadata.get('aiapy_version', '?')}")
print(f"Time correction / EVE normalization: {response_metadata.get('time_dependent_correction', False)} / {response_metadata.get('eve_normalization', False)}")

## Solver adapters

각 solver를 ROI의 모든 픽셀에 대해 실행하고, 나온 DEM들을 픽셀 평균해 profile 한 개로 만든다. 모든 profile은 **DEM(T) [cm⁻⁵ K⁻¹]** convention으로 통일한다 (per-dex 출력은 dlogT/dT = 1/(T·ln10) 인자로 변환). 현재 `2_Solver`에서 **AIA 6채널 응답으로 완전한 Python 역산을 실행할 수 있는 5개**를 포함한다.

- `simple_reg_dem`: (ny, nx, channel) 맵 입력을 그대로 받는다. DN count와 exposure 사용. 출력이 DEM(logT) [cm⁻⁵ dex⁻¹]라 1/(T·ln10)을 곱해 DEM(T)로 변환한다.
- `demreg`: Hannah & Kontar GSVD 정규화. DN/s와 오차를 사용하며, 출력이 이미 DEM(T) [cm⁻⁵ K⁻¹]라 변환 없이 사용한다. 검증된 경로인 `dem_norm0=ones`를 사용하고 notebook multiprocessing 문제를 피하려고 128픽셀씩 처리한다.
- `dem_sites`: 단일 픽셀 전용이라 픽셀 루프를 돈다. 수정된 IDL 폭 규칙의 기본 Gaussian 커널(현재 41-bin 격자에서는 23 points)을 한 번 만들어 모든 픽셀에 재사용하고, 5% 잔차에서 조기 종료한다. `delta_temp` 인자에 ΔlogT를 전달하므로 출력은 DEM(logT)이고 1/(T·ln10)을 곱해 DEM(T)로 변환한다.
- `cheung_sparse_em`: 맵 입력 지원. 선택한 LP 엔진을 명시적으로 전달하며, IDL 규약상 `status=0`인 픽셀만 평균한다. `oem`은 bin당 EM[cm⁻⁵]이라 ΔlogT로 나누고 1/(T·ln10)을 곱해 DEM(T)로 변환한다. 엔진명·성공률·상태 코드별 픽셀 수를 출력해 비수렴을 숨기지 않는다.
- `aschwanden_aia_teem`: 맵 입력 지원. 픽셀별 단일 Gaussian fit 진폭이 DEM(T)[cm⁻⁵ K⁻¹]라 변환 없이 사용하고, fit 성공 픽셀만 평균.

비교에서 제외한 패키지: `firdems`는 Python 변환본에 최종 `firdem_iterate`가 없어 first-pass만 가능하고, `trace_dem`은 TRACE 3채널 전용, `eit_dem`은 EIT 전용 보정, `chianti_dem`·`pintofale_mcmc`는 스펙트럼 선 입력용, `vdem`은 속도 DEM, `xrt_iterative`는 DEM을 찾는 최적화 루프가 없다.

In [ ]:
# 비교 convention: 모든 profile은 DEM(T) [cm^-5 K^-1]로 통일한다.
# per-dex(DEM(logT)) 출력 solver는 dlogT/dT = 1/(T·ln10) 인자를 곱해 변환한다.
temperature_kelvin = 10.0 ** logt
dem_logt_to_dem_t = 1.0 / (temperature_kelvin * np.log(10.0))


def run_simple_reg_dem():
    dem_map, chi2_map = simple_reg_dem(
        roi_counts,
        roi_sigma_counts,
        exptime,
        logt,
        temperature_response,
    )
    # simple_reg_dem 출력은 DEM(logT) [cm^-5 dex^-1] (내부 행렬이 ΔlogT 기반) —
    # 1/(T·ln10)을 곱해 DEM(T) [cm^-5 K^-1]로 변환한다.
    return {
        'profile': np.mean(dem_map, axis=(0, 1), dtype=np.float64) * dem_logt_to_dem_t,
        'diagnostics': {'mean_chi2': float(np.mean(chi2_map))},
    }


def run_demreg():
    if not 0 < DEMREG_BATCH_PIXELS < 256:
        raise ValueError('DEMREG_BATCH_PIXELS는 Jupyter 안전성을 위해 1..255 범위여야 합니다.')

    pixel_rate = roi_rate.reshape(-1, wavelengths.size)
    pixel_sigma_rate = roi_sigma_rate.reshape(-1, wavelengths.size)
    logt_edges = np.empty(logt.size + 1, dtype=np.float64)
    logt_edges[1:-1] = 0.5 * (logt[:-1] + logt[1:])
    logt_edges[0] = logt[0] - 0.5 * (logt[1] - logt[0])
    logt_edges[-1] = logt[-1] + 0.5 * (logt[-1] - logt[-2])
    temperature_edges = 10.0 ** logt_edges

    dem_t_batches = []
    chisq_batches = []
    negative_pixels = 0
    for start in range(0, pixel_rate.shape[0], DEMREG_BATCH_PIXELS):
        stop = min(start + DEMREG_BATCH_PIXELS, pixel_rate.shape[0])
        batch_size = stop - start
        # dn2dem_pos는 256픽셀 이상에서 ProcessPoolExecutor를 사용한다. Notebook/Windows에서
        # 안전하게 실행되도록 작은 serial batch로 나누고 내부의 반복 timing 출력은 숨긴다.
        with contextlib.redirect_stdout(io.StringIO()):
            dem_t, _, _, chisq, _ = dn2dem_pos(
                pixel_rate[start:stop],
                pixel_sigma_rate[start:stop],
                temperature_response,
                logt,
                temperature_edges,
                dem_norm0=np.ones((batch_size, logt.size), dtype=np.float64),
            )
        dem_t = np.asarray(dem_t, dtype=np.float64).reshape(batch_size, logt.size)
        chisq = np.asarray(chisq, dtype=np.float64).reshape(batch_size)
        if not np.all(np.isfinite(dem_t)) or not np.all(np.isfinite(chisq)):
            raise ValueError(f'demreg: non-finite result in pixels {start}:{stop}')
        negative_pixels += int(np.count_nonzero(np.any(dem_t < 0.0, axis=1)))
        dem_t_batches.append(dem_t)
        chisq_batches.append(chisq)

    # dn2dem_pos 출력은 이미 DEM(T) [cm^-5 K^-1] — 비교 convention과 같아 변환하지 않는다.
    dem_t_all = np.concatenate(dem_t_batches, axis=0)
    chisq = np.concatenate(chisq_batches)
    return {
        'profile': dem_t_all.mean(axis=0),
        'diagnostics': {
            'mean_chi2': float(chisq.mean()),
            'negative_pixels': negative_pixels,
        },
    }


def run_dem_sites():
    ny, nx = roi_rate.shape[:2]
    dems = np.empty((ny, nx, logt.size))
    iterations = np.empty((ny, nx))
    response_error = np.full(wavelengths.size, SITES_RESPONSE_ERROR, dtype=np.float64)
    delta_logt = np.gradient(logt)
    sites_sigma = max(
        np.float32(np.float32(0.08) * np.float32(logt.size)),
        np.float32(0.5),
    )
    sites_kernel = gaussian_function(sites_sigma)

    for i in range(ny):
        for j in range(nx):
            dems[i, j], _, _, iterations[i, j] = dem_sites(
                roi_rate[i, j],
                roi_sigma_rate[i, j],
                temperature_response,
                response_error,
                delta_logt,
                convergence=0.05,
                ker=sites_kernel,
            )
    # delta_temp 인자로 ΔlogT를 전달했으므로 출력은 DEM(logT) [cm^-5 dex^-1] —
    # 1/(T·ln10)을 곱해 DEM(T) [cm^-5 K^-1]로 변환한다.
    return {
        'profile': dems.mean(axis=(0, 1)) * dem_logt_to_dem_t,
        'diagnostics': {
            'mean_iterations': float(iterations.mean()),
            'kernel_points': int(sites_kernel.size),
        },
    }


def run_cheung_sparse_em():
    if CHEUNG_ENGINE not in {'idl', 'highs'}:
        raise ValueError("CHEUNG_ENGINE은 'idl' 또는 'highs'여야 합니다.")

    dictionary, basis_functions = aia_sparse_em_init(
        temperature_response.T,
        logt,
        bases_sigmas=(0.0, 0.1, 0.2),
        dictfac=1e26,
    )
    _, oem, _, status = aia_sparse_em_solve(
        roi_rate,
        dictionary,
        basis_functions,
        tolfac=1.4,
        engine=CHEUNG_ENGINE,
        itmax=CHEUNG_ITMAX,
    )
    status_int = np.asarray(status, dtype=np.int16)
    ok = status_int == 0
    status_counts = {
        f'status_{code}': int(np.count_nonzero(status_int == code))
        for code in (0, 1, 2, 3, 10, 11)
    }
    if not np.any(ok):
        raise RuntimeError(
            f'cheung_sparse_em: 성공한 픽셀이 없습니다 ({status_counts}).'
        )
    # oem은 온도 bin당 EM [cm^-5]이다 (Dict에 ΔlogT 적분이 없음).
    # ΔlogT로 나눠 DEM(logT)로 만든 뒤 1/(T·ln10)을 곱해
    # 비교 convention인 DEM(T) [cm^-5 K^-1]로 변환한다.
    delta_logt = logt[1] - logt[0]
    return {
        'profile': np.mean(oem[ok, :], axis=0) * 1e26 / delta_logt * dem_logt_to_dem_t,
        'diagnostics': {
            'engine': CHEUNG_ENGINE,
            'success_fraction': float(np.mean(ok)),
            **status_counts,
        },
    }


def run_aschwanden_aia_teem():
    temperature_kelvin = 10.0 ** logt
    delta_kelvin = np.diff(temperature_kelvin)
    delta_kelvin = np.concatenate([delta_kelvin, delta_kelvin[-1:]])
    sigma_grid = np.linspace(0.05, 0.40, 12)
    flux_table = build_flux_table(temperature_response, logt, delta_kelvin, sigma_grid)
    te_map, log_em_map, sigma_map, chi_map = aia_teem_fit(
        roi_rate, exptime, flux_table, logt, sigma_grid
    )
    # log_em은 DEM(T) [cm^-5 K^-1] Gaussian 진폭이다 (flux table이 ΔT[K]로 적분됨).
    # 비교 convention이 DEM(T)라서 변환 없이 그대로 사용한다.
    # sigma=0은 fit 실패 픽셀이므로 평균에서 제외한다.
    fitted = sigma_map > 0
    if not np.any(fitted):
        raise RuntimeError('aschwanden_aia_teem: 성공한 픽셀이 없습니다.')
    te = te_map[fitted][:, None]
    sig = sigma_map[fitted][:, None]
    log_em = log_em_map[fitted][:, None]
    dem_t = 10.0 ** log_em * np.exp(-0.5 * ((logt[None, :] - te) / sig) ** 2)
    return {
        'profile': dem_t.mean(axis=0),
        'diagnostics': {
            'fitted_fraction': float(np.mean(fitted)),
            'mean_peak_logT': float(te_map[fitted].mean()),
            'mean_chi': float(chi_map[fitted].mean()),
        },
    }


SOLVER_RUNNERS = {
    'simple_reg_dem': run_simple_reg_dem,
    'demreg': run_demreg,
    'dem_sites': run_dem_sites,
    'cheung_sparse_em': run_cheung_sparse_em,
    'aschwanden_aia_teem': run_aschwanden_aia_teem,
}

unknown_solvers = sorted(set(SELECTED_SOLVERS) - set(SOLVER_RUNNERS))
if unknown_solvers:
    raise KeyError(f'알 수 없는 solver: {unknown_solvers}; 선택 가능: {sorted(SOLVER_RUNNERS)}')
if not SELECTED_SOLVERS:
    raise ValueError('SELECTED_SOLVERS에서 하나 이상의 solver를 선택하세요.')

_trapezoid = getattr(np, 'trapezoid', None)
if _trapezoid is None:
    _trapezoid = np.trapz


def _format_diagnostic(value):
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.5g}'
    return str(value)


results = {}
for solver_name in SELECTED_SOLVERS:
    started = time.perf_counter()
    result = SOLVER_RUNNERS[solver_name]()
    profile = np.maximum(np.asarray(result['profile'], dtype=np.float64), 0.0)
    if profile.shape != logt.shape or not np.all(np.isfinite(profile)):
        raise ValueError(f'{solver_name}: 유효하지 않은 profile shape/value')
    # total EM = ∫ DEM(T) dT [cm^-5] — T축 적분
    area = float(_trapezoid(profile, temperature_kelvin))
    if not np.isfinite(area) or area <= 0:
        raise ValueError(f'{solver_name}: profile 적분값이 양수가 아닙니다: {area}')
    result['profile'] = profile
    result['normalized_profile'] = profile / area
    result['elapsed_seconds'] = time.perf_counter() - started
    results[solver_name] = result

for solver_name, result in results.items():
    diag = ', '.join(
        f'{key}={_format_diagnostic(value)}'
        for key, value in result['diagnostics'].items()
    )
    print(f"{solver_name:24s} {result['elapsed_seconds']:.3f} s | {diag}")

In [ ]:
display_labels = {
    'simple_reg_dem': 'Simple regularized DEM',
    'demreg': 'Hannah & Kontar regularized DEM',
    'dem_sites': 'SITES',
    'cheung_sparse_em': 'Cheung sparse EM',
    'aschwanden_aia_teem': 'Aschwanden single Gaussian',
}

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
for solver_name, result in results.items():
    ax.plot(logt, result['profile'], linewidth=2, label=display_labels[solver_name])
ax.set_yscale('log')
profile_peak = max(result['profile'].max() for result in results.values())
ax.set_ylim(profile_peak * 1e-6, profile_peak * 3)
ax.set_xlabel(r'$\log_{10}(T\,[\mathrm{K}])$')
ax.set_ylabel(r'$\mathrm{DEM}(T)\ [\mathrm{cm^{-5}}\,\mathrm{K^{-1}}]$')
ax.set_title('DEM profiles — tabulated AIA temperature response')
ax.set_xlim(logt[0], logt[-1])
ax.grid(alpha=0.25, which='both')
ax.legend(frameon=False)
plt.show()

## 다음 단계

현재 실제 AIA TRF를 적용했지만 관측시각별 성능저하, EVE 정규화, 94/131 Å CHIANTI 보정은 포함하지 않았다. 각 solver의 DEM convention은 DEM(T) [cm⁻⁵ K⁻¹]로 통일했다. 정량 분석에서는 해당 옵션으로 응답을 다시 생성하고, intensity가 DN인지 DN/s인지 확인해야 한다.